# Smart Packaging Line — 8-Model Comparison Toward Six Sigma (99.9997%)

**Project:** Smart Packaging Line for TE Connectivity CPT MX  
**Goal:** Weight-based kit inspection with 99.9997% accuracy (Six Sigma level)  
**Data:** 420 kits across recipes R3, R4, R6  
**Models:** Logistic Regression, LDA, QDA, GNB, Random Forest, SVM, BLR, CUSUM

---

## 0. Setup: Load Data from Database

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Database path
DB_PATH = "/home/claude/smart_packaging.db"

def load_data_from_db(db_path):
    """Load inspection data from SQLite database."""
    conn = sqlite3.connect(db_path)
    
    query = """
    SELECT 
        i.inspection_id,
        i.run_id,
        i.recipe_id,
        i.measured_weight_g,
        i.ground_truth_label,
        i.missing_component,
        r.expected_weight_g,
        r.expected_std_g,
        r.lower_bound_3s,
        r.upper_bound_3s
    FROM inspections i
    JOIN recipes r ON i.recipe_id = r.recipe_id
    ORDER BY i.run_id
    """
    
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    return df

# Load data
df = load_data_from_db(DB_PATH)

print(f"✅ Data Loaded")
print(f"   Shape: {df.shape}")
print(f"   Recipes: {df['recipe_id'].unique()}")
print(f"\nGround Truth Distribution:")
print(df['ground_truth_label'].value_counts())
print(f"\nFirst 5 rows:")
print(df.head())

## 1. Data Preparation & Feature Engineering

In [ ]:
# Prepare features and labels
X = df[['measured_weight_g']].values  # Feature: weight
y = (df['ground_truth_label'] == 'OK').astype(int).values  # Label: 1=OK, 0=NG

# Split by recipe for stratified analysis
recipes = df['recipe_id'].unique()

print(f"Feature matrix shape: {X.shape}")
print(f"Label distribution: {np.bincount(y)}")
print(f"  OK (1): {np.sum(y)} ({100*np.sum(y)/len(y):.1f}%)")
print(f"  NG (0): {len(y) - np.sum(y)} ({100*(len(y)-np.sum(y))/len(y):.1f}%)")

# Weight statistics
print(f"\nWeight statistics:")
print(f"  Mean: {X.mean():.3f}g")
print(f"  Std:  {X.std():.3f}g")
print(f"  Min:  {X.min():.3f}g")
print(f"  Max:  {X.max():.3f}g")

## 2. Baseline: Your Current Z-Score Method

In [ ]:
def zscore_classifier(weights, expected_w, sigma_w, k=3.0):
    """Your current z-score Level 1 classifier."""
    z = np.abs((weights - expected_w) / sigma_w)
    return (z <= k).astype(int)  # 1=OK, 0=NG

# Apply z-score per recipe
y_pred_zscore = np.zeros_like(y)

for idx, row in df.iterrows():
    pred = zscore_classifier(
        row['measured_weight_g'],
        row['expected_weight_g'],
        row['expected_std_g'],
        k=3.0
    )
    y_pred_zscore[idx] = pred

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

zscore_acc = accuracy_score(y, y_pred_zscore)
zscore_prec = precision_score(y, y_pred_zscore)
zscore_rec = recall_score(y, y_pred_zscore)
zscore_f1 = f1_score(y, y_pred_zscore)

print("="*60)
print("BASELINE: Z-Score (k=3.0)")
print("="*60)
print(f"Accuracy:  {zscore_acc:.6f}")
print(f"Precision: {zscore_prec:.6f}")
print(f"Recall:    {zscore_rec:.6f}")
print(f"F1-Score:  {zscore_f1:.6f}")

## 3. Model 1: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Train Logistic Regression
lr = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
lr.fit(X, y)

# Predictions
y_pred_lr = lr.predict(X)
y_pred_lr_proba = lr.predict_proba(X)[:, 1]

# Cross-validation
cv_scores_lr = cross_val_score(lr, X, y, cv=5, scoring='accuracy')

# Metrics
lr_acc = accuracy_score(y, y_pred_lr)
lr_prec = precision_score(y, y_pred_lr)
lr_rec = recall_score(y, y_pred_lr)
lr_f1 = f1_score(y, y_pred_lr)
lr_auc = roc_auc_score(y, y_pred_lr_proba)

print("="*60)
print("MODEL 1: Logistic Regression")
print("="*60)
print(f"Accuracy:  {lr_acc:.6f}")
print(f"Precision: {lr_prec:.6f}")
print(f"Recall:    {lr_rec:.6f}")
print(f"F1-Score:  {lr_f1:.6f}")
print(f"AUC-ROC:   {lr_auc:.6f}")
print(f"CV Accuracy: {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std():.4f})")
print(f"\nModel Parameters:")
print(f"  β₀ (intercept): {lr.intercept_[0]:.4f}")
print(f"  β₁ (weight coef): {lr.coef_[0][0]:.4f}")
threshold_lr = -lr.intercept_[0] / lr.coef_[0][0]
print(f"  Threshold w*: {threshold_lr:.3f}g")

## 4. Model 2: Linear Discriminant Analysis (LDA)

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from scipy.stats import shapiro, levene

# Test assumptions
X_ok = X[y == 1]
X_ng = X[y == 0]

shapiro_ok = shapiro(X_ok)
shapiro_ng = shapiro(X_ng)
levene_test = levene(X_ok, X_ng)

print("Assumption Tests:")
print(f"  Shapiro-Wilk (OK): p={shapiro_ok.pvalue:.4f} (Normal: {shapiro_ok.pvalue > 0.05})")
print(f"  Shapiro-Wilk (NG): p={shapiro_ng.pvalue:.4f} (Normal: {shapiro_ng.pvalue > 0.05})")
print(f"  Levene (σ_OK = σ_NG): p={levene_test.pvalue:.4f} (Equal: {levene_test.pvalue > 0.05})")

# Train LDA
lda = LinearDiscriminantAnalysis(solver='svd')
lda.fit(X, y)

# Predictions
y_pred_lda = lda.predict(X)
y_pred_lda_proba = lda.predict_proba(X)[:, 1]

# Cross-validation
cv_scores_lda = cross_val_score(lda, X, y, cv=5, scoring='accuracy')

# Metrics
lda_acc = accuracy_score(y, y_pred_lda)
lda_prec = precision_score(y, y_pred_lda)
lda_rec = recall_score(y, y_pred_lda)
lda_f1 = f1_score(y, y_pred_lda)
lda_auc = roc_auc_score(y, y_pred_lda_proba)

print("\n" + "="*60)
print("MODEL 2: Linear Discriminant Analysis (LDA)")
print("="*60)
print(f"Accuracy:  {lda_acc:.6f}")
print(f"Precision: {lda_prec:.6f}")
print(f"Recall:    {lda_rec:.6f}")
print(f"F1-Score:  {lda_f1:.6f}")
print(f"AUC-ROC:   {lda_auc:.6f}")
print(f"CV Accuracy: {cv_scores_lda.mean():.4f} (+/- {cv_scores_lda.std():.4f})")
print(f"\nEstimated Parameters:")
print(f"  μ_OK:  {lda.means_[1][0]:.3f}g")
print(f"  μ_NG:  {lda.means_[0][0]:.3f}g")
print(f"  σ_pooled: {np.sqrt(lda.covariance_[0, 0]):.4f}g")
threshold_lda = (lda.means_[0][0] + lda.means_[1][0]) / 2
print(f"  Threshold w*: {threshold_lda:.3f}g")

## 5. Model 3: Quadratic Discriminant Analysis (QDA)

In [ ]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

# Train QDA
qda = QuadraticDiscriminantAnalysis(store_covariance=True)
qda.fit(X, y)

# Predictions
y_pred_qda = qda.predict(X)
y_pred_qda_proba = qda.predict_proba(X)[:, 1]

# Cross-validation
cv_scores_qda = cross_val_score(qda, X, y, cv=5, scoring='accuracy')

# Metrics
qda_acc = accuracy_score(y, y_pred_qda)
qda_prec = precision_score(y, y_pred_qda)
qda_rec = recall_score(y, y_pred_qda)
qda_f1 = f1_score(y, y_pred_qda)
qda_auc = roc_auc_score(y, y_pred_qda_proba)

print("="*60)
print("MODEL 3: Quadratic Discriminant Analysis (QDA)")
print("="*60)
print(f"Accuracy:  {qda_acc:.6f}")
print(f"Precision: {qda_prec:.6f}")
print(f"Recall:    {qda_rec:.6f}")
print(f"F1-Score:  {qda_f1:.6f}")
print(f"AUC-ROC:   {qda_auc:.6f}")
print(f"CV Accuracy: {cv_scores_qda.mean():.4f} (+/- {cv_scores_qda.std():.4f})")
print(f"\nEstimated Parameters:")
print(f"  σ_OK:  {np.sqrt(qda.covariance_[1, 0, 0]):.4f}g")
print(f"  σ_NG:  {np.sqrt(qda.covariance_[0, 0, 0]):.4f}g")
print(f"  Ratio σ_NG/σ_OK: {np.sqrt(qda.covariance_[0, 0, 0]) / np.sqrt(qda.covariance_[1, 0, 0]):.2f}x")

## 6. Model 4: Gaussian Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

# Train GNB
gnb = GaussianNB()
gnb.fit(X, y)

# Predictions
y_pred_gnb = gnb.predict(X)
y_pred_gnb_proba = gnb.predict_proba(X)[:, 1]

# Cross-validation
cv_scores_gnb = cross_val_score(gnb, X, y, cv=5, scoring='accuracy')

# Metrics
gnb_acc = accuracy_score(y, y_pred_gnb)
gnb_prec = precision_score(y, y_pred_gnb)
gnb_rec = recall_score(y, y_pred_gnb)
gnb_f1 = f1_score(y, y_pred_gnb)
gnb_auc = roc_auc_score(y, y_pred_gnb_proba)

print("="*60)
print("MODEL 4: Gaussian Naive Bayes (GNB)")
print("="*60)
print(f"Accuracy:  {gnb_acc:.6f}")
print(f"Precision: {gnb_prec:.6f}")
print(f"Recall:    {gnb_rec:.6f}")
print(f"F1-Score:  {gnb_f1:.6f}")
print(f"AUC-ROC:   {gnb_auc:.6f}")
print(f"CV Accuracy: {cv_scores_gnb.mean():.4f} (+/- {cv_scores_gnb.std():.4f})")
print(f"\nNote: With 1 variable, GNB is mathematically equivalent to LDA/QDA")

## 7. Model 5: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest (default hyperparameters)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10)
rf.fit(X, y)

# Predictions
y_pred_rf = rf.predict(X)
y_pred_rf_proba = rf.predict_proba(X)[:, 1]

# Cross-validation
cv_scores_rf = cross_val_score(rf, X, y, cv=5, scoring='accuracy')

# Metrics
rf_acc = accuracy_score(y, y_pred_rf)
rf_prec = precision_score(y, y_pred_rf)
rf_rec = recall_score(y, y_pred_rf)
rf_f1 = f1_score(y, y_pred_rf)
rf_auc = roc_auc_score(y, y_pred_rf_proba)

print("="*60)
print("MODEL 5: Random Forest (100 trees, max_depth=10)")
print("="*60)
print(f"Accuracy:  {rf_acc:.6f}")
print(f"Precision: {rf_prec:.6f}")
print(f"Recall:    {rf_rec:.6f}")
print(f"F1-Score:  {rf_f1:.6f}")
print(f"AUC-ROC:   {rf_auc:.6f}")
print(f"CV Accuracy: {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std():.4f})")
print(f"\nFeature Importance:")
print(f"  Weight: {rf.feature_importances_[0]:.4f}")

## 8. Model 6: Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# Standardize data (critical for SVM)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train SVM (linear kernel)
svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm.fit(X_scaled, y)

# Predictions
y_pred_svm = svm.predict(X_scaled)
y_pred_svm_proba = svm.predict_proba(X_scaled)[:, 1]

# Cross-validation
cv_scores_svm = cross_val_score(svm, X_scaled, y, cv=5, scoring='accuracy')

# Metrics
svm_acc = accuracy_score(y, y_pred_svm)
svm_prec = precision_score(y, y_pred_svm)
svm_rec = recall_score(y, y_pred_svm)
svm_f1 = f1_score(y, y_pred_svm)
svm_auc = roc_auc_score(y, y_pred_svm_proba)

print("="*60)
print("MODEL 6: Support Vector Machine (SVM RBF)")
print("="*60)
print(f"Accuracy:  {svm_acc:.6f}")
print(f"Precision: {svm_prec:.6f}")
print(f"Recall:    {svm_rec:.6f}")
print(f"F1-Score:  {svm_f1:.6f}")
print(f"AUC-ROC:   {svm_auc:.6f}")
print(f"CV Accuracy: {cv_scores_svm.mean():.4f} (+/- {cv_scores_svm.std():.4f})")
print(f"\nModel Parameters:")
print(f"  n_support_vectors: {len(svm.support_)}")
print(f"  Kernel: RBF (γ=scale)")

## 9. Model 7: Bayesian Logistic Regression (BLR) using PyMC

In [ ]:
try:
    import pymc as pm
    import arviz as az
    
    print("Building Bayesian Logistic Regression model...")
    print("(This may take 30-60 seconds)\n")
    
    # Define PyMC model
    with pm.Model() as model_blr:
        # Priors
        β0 = pm.Normal('β0', mu=0, sigma=10)
        β1 = pm.Normal('β1', mu=0, sigma=10)
        
        # Logistic model
        p = pm.math.sigmoid(β0 + β1 * X.flatten())
        
        # Likelihood
        y_obs = pm.Bernoulli('y_obs', p=p, observed=y)
        
        # Posterior sampling
        trace = pm.sample(2000, tune=2000, cores=4, return_inferencedata=True, 
                          progressbar=False, random_seed=42)
    
    # Get posterior samples for predictions
    posterior_β0 = trace.posterior['β0'].values.flatten()
    posterior_β1 = trace.posterior['β1'].values.flatten()
    
    # Posterior predictive: P(OK | w)
    y_pred_blr_proba = np.array([
        (1 / (1 + np.exp(-(posterior_β0.mean() + posterior_β1.mean() * w)))) 
        for w in X.flatten()
    ])
    y_pred_blr = (y_pred_blr_proba > 0.5).astype(int)
    
    # Metrics
    blr_acc = accuracy_score(y, y_pred_blr)
    blr_prec = precision_score(y, y_pred_blr)
    blr_rec = recall_score(y, y_pred_blr)
    blr_f1 = f1_score(y, y_pred_blr)
    blr_auc = roc_auc_score(y, y_pred_blr_proba)
    
    print("="*60)
    print("MODEL 7: Bayesian Logistic Regression (PyMC)")
    print("="*60)
    print(f"Accuracy:  {blr_acc:.6f}")
    print(f"Precision: {blr_prec:.6f}")
    print(f"Recall:    {blr_rec:.6f}")
    print(f"F1-Score:  {blr_f1:.6f}")
    print(f"AUC-ROC:   {blr_auc:.6f}")
    print(f"\nPosterior Estimates (mean ± std):")
    print(f"  β₀: {posterior_β0.mean():.4f} ± {posterior_β0.std():.4f}")
    print(f"  β₁: {posterior_β1.mean():.4f} ± {posterior_β1.std():.4f}")
    print(f"\n95% Credible Intervals (HDI):")
    print(f"  β₀: [{np.percentile(posterior_β0, 2.5):.4f}, {np.percentile(posterior_β0, 97.5):.4f}]")
    print(f"  β₁: [{np.percentile(posterior_β1, 2.5):.4f}, {np.percentile(posterior_β1, 97.5):.4f}]")
    
except ImportError:
    print("PyMC not installed. Skipping BLR...")
    print("Install with: pip install pymc\n")
    
    # Use Logistic Regression as proxy
    blr_acc = lr_acc
    blr_prec = lr_prec
    blr_rec = lr_rec
    blr_f1 = lr_f1
    blr_auc = lr_auc

## 10. Model 8: CUSUM (Change-Point Detection)

In [ ]:
def cusum_control_chart(weights, target, k=0.5, h=5, sigma=0.04):
    """
    CUSUM (Cumulative Sum) control chart for drift detection.
    Not a classifier, but a monitoring tool.
    """
    cusum = np.zeros(len(weights))
    alarms = []
    
    for t in range(len(weights)):
        deviation = weights[t] - target - k*sigma
        cusum[t] = max(0, (cusum[t-1] if t > 0 else 0) + deviation)
        
        if cusum[t] > h*sigma:
            alarms.append(t)
    
    return cusum, alarms

# Apply CUSUM per recipe (monitoring only, not classification)
print("="*60)
print("MODEL 8: CUSUM (Quality Control Chart)")
print("="*60)
print(f"Purpose: Detect process drift, not classify individual kits")
print(f"Status: Post-validation analysis (after 420-kit completion)")
print(f"\nCUSUM is not a classifier, it's a monitoring tool.")
print(f"Use to detect when scale calibration drifts over time.")

# Set placeholder metrics for compatibility
cusum_acc = 0.0
cusum_auc = 0.0

---
# PART II: STANDARDIZED COMPARISON & SIX SIGMA ANALYSIS

## 11. Comprehensive Model Comparison Table

In [ ]:
# Compile all results
results = pd.DataFrame({
    'Model': ['Z-Score (Baseline)', 'Logistic Regression', 'LDA', 'QDA', 'Gaussian NB', 
              'Random Forest', 'SVM (RBF)', 'Bayesian LR', 'CUSUM'],
    'Accuracy': [zscore_acc, lr_acc, lda_acc, qda_acc, gnb_acc, rf_acc, svm_acc, blr_acc, np.nan],
    'Precision': [zscore_prec, lr_prec, lda_prec, qda_prec, gnb_prec, rf_prec, svm_prec, blr_prec, np.nan],
    'Recall': [zscore_rec, lr_rec, lda_rec, qda_rec, gnb_rec, rf_rec, svm_rec, blr_rec, np.nan],
    'F1-Score': [zscore_f1, lr_f1, lda_f1, qda_f1, gnb_f1, rf_f1, svm_f1, blr_f1, np.nan],
    'AUC-ROC': [np.nan, lr_auc, lda_auc, qda_auc, gnb_auc, rf_auc, svm_auc, blr_auc, np.nan],
})

print("\n" + "="*100)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*100)
print(results.to_string(index=False))

# Rank by Accuracy
results_ranked = results.dropna(subset=['Accuracy']).sort_values('Accuracy', ascending=False).reset_index(drop=True)
results_ranked['Rank'] = range(1, len(results_ranked) + 1)

print(f"\n\nRANKING BY ACCURACY:")
print(results_ranked[['Rank', 'Model', 'Accuracy', 'AUC-ROC']].to_string(index=False))

## 12. Six Sigma Target Analysis (99.9997%)

In [ ]:
# Six Sigma target
SIX_SIGMA_TARGET = 0.999997
DEFECTS_PER_MILLION = 3.4  # DPMO for Six Sigma

# Convert accuracies to DPMO
def accuracy_to_dpmo(accuracy):
    """Convert accuracy to Defects Per Million Opportunities."""
    if pd.isna(accuracy):
        return np.nan
    return (1 - accuracy) * 1e6

# Add DPMO column
results['DPMO'] = results['Accuracy'].apply(accuracy_to_dpmo)
results['vs Six Sigma'] = results['DPMO'].apply(
    lambda x: '✓ PASS' if pd.notna(x) and x <= DEFECTS_PER_MILLION else ('✗ FAIL' if pd.notna(x) else 'N/A')
)

print("\n" + "="*100)
print(f"SIX SIGMA ANALYSIS (Target: {SIX_SIGMA_TARGET:.6f}, DPMO ≤ {DEFECTS_PER_MILLION:.1f})")
print("="*100)
print(f"\n{'Model':<25} {'Accuracy':<15} {'DPMO':<15} {'Status'}")
print("-" * 70)

for idx, row in results.iterrows():
    model = row['Model']
    acc = row['Accuracy']
    dpmo = row['DPMO']
    status = row['vs Six Sigma']
    
    if pd.isna(acc):
        print(f"{model:<25} {'N/A':<15} {'N/A':<15} {status}")
    else:
        print(f"{model:<25} {acc:.6f}       {dpmo:>10.1f}     {status}")

# Count passes
passes = (results['vs Six Sigma'] == '✓ PASS').sum()
print(f"\n→ {passes} out of {len(results)-1} classifiers achieve Six Sigma")

## 13. Per-Recipe Performance Breakdown

In [ ]:
# Analyze by recipe
print("\n" + "="*100)
print("PERFORMANCE BY RECIPE")
print("="*100)

recipe_performance = {}

for recipe in ['R3', 'R4', 'R6']:
    mask = df['recipe_id'] == recipe
    
    y_recipe = y[mask]
    X_recipe = X[mask]
    
    print(f"\n{recipe}:")
    print(f"  Samples: {len(y_recipe)} (OK: {np.sum(y_recipe)}, NG: {len(y_recipe) - np.sum(y_recipe)})")
    print(f"  Weight range: {X_recipe.min():.3f}g - {X_recipe.max():.3f}g")
    print(f"  Accuracy by model:")
    
    # Per-recipe accuracies
    print(f"    Z-Score:  {accuracy_score(y_recipe, y_pred_zscore[mask]):.4f}")
    print(f"    Logistic: {accuracy_score(y_recipe, y_pred_lr[mask]):.4f}")
    print(f"    LDA:      {accuracy_score(y_recipe, y_pred_lda[mask]):.4f}")
    print(f"    QDA:      {accuracy_score(y_recipe, y_pred_qda[mask]):.4f}")
    print(f"    RF:       {accuracy_score(y_recipe, y_pred_rf[mask]):.4f}")
    print(f"    SVM:      {accuracy_score(y_recipe, y_pred_svm[mask]):.4f}")

## 14. Visualization: Model Comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create comparison visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Smart Packaging Line — 8-Model Comparison', fontsize=16, fontweight='bold')

# 1. Accuracy comparison
ax1 = axes[0, 0]
results_plot = results.dropna(subset=['Accuracy']).sort_values('Accuracy', ascending=True)
colors = ['green' if x == '✓ PASS' else 'red' for x in results_plot['vs Six Sigma']]
ax1.barh(results_plot['Model'], results_plot['Accuracy'], color=colors, alpha=0.7)
ax1.axvline(SIX_SIGMA_TARGET, color='darkgreen', linestyle='--', linewidth=2, label='Six Sigma Target')
ax1.axvline(zscore_acc, color='blue', linestyle=':', linewidth=2, label='Baseline (Z-Score)')
ax1.set_xlabel('Accuracy')
ax1.set_title('Accuracy Ranking')
ax1.legend()
ax1.set_xlim([0.98, 1.0001])
for i, v in enumerate(results_plot['Accuracy']):
    ax1.text(v + 0.0001, i, f'{v:.6f}', va='center', fontsize=9)

# 2. AUC-ROC comparison
ax2 = axes[0, 1]
results_auc = results.dropna(subset=['AUC-ROC']).sort_values('AUC-ROC', ascending=True)
ax2.barh(results_auc['Model'], results_auc['AUC-ROC'], color='steelblue', alpha=0.7)
ax2.axvline(0.99, color='darkgreen', linestyle='--', linewidth=2, label='High Performance')
ax2.set_xlabel('AUC-ROC')
ax2.set_title('AUC-ROC Ranking')
ax2.legend()
ax2.set_xlim([0.95, 1.01])

# 3. Precision-Recall scatter
ax3 = axes[1, 0]
results_pr = results.dropna(subset=['Precision', 'Recall'])
ax3.scatter(results_pr['Recall'], results_pr['Precision'], s=200, alpha=0.6)
for idx, row in results_pr.iterrows():
    ax3.annotate(row['Model'].split('(')[0].strip(), 
                (row['Recall'], row['Precision']), 
                fontsize=8, ha='right')
ax3.set_xlabel('Recall')
ax3.set_ylabel('Precision')
ax3.set_title('Precision-Recall Trade-off')
ax3.set_xlim([0.95, 1.01])
ax3.set_ylim([0.95, 1.01])
ax3.grid(True, alpha=0.3)

# 4. DPMO vs Six Sigma
ax4 = axes[1, 1]
results_dpmo = results.dropna(subset=['DPMO']).sort_values('DPMO', ascending=True)
colors_dpmo = ['green' if x <= DEFECTS_PER_MILLION else 'red' for x in results_dpmo['DPMO']]
ax4.barh(results_dpmo['Model'], results_dpmo['DPMO'], color=colors_dpmo, alpha=0.7)
ax4.axvline(DEFECTS_PER_MILLION, color='darkgreen', linestyle='--', linewidth=2, label=f'Six Sigma: {DEFECTS_PER_MILLION:.1f} DPMO')
ax4.set_xlabel('Defects Per Million (DPMO)')
ax4.set_title('DPMO vs Six Sigma Target')
ax4.set_xscale('log')
ax4.legend()

plt.tight_layout()
plt.savefig('/home/claude/model_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Visualization saved: /home/claude/model_comparison.png")
plt.show()

## 15. Feature Importance & Decision Boundaries

In [ ]:
# Plot decision boundaries and weight distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Weight Distributions & Classification Boundaries', fontsize=14, fontweight='bold')

# 1. Weight distributions by class
ax1 = axes[0]
ax1.hist(X[y == 1], bins=30, alpha=0.6, label='OK', color='green', edgecolor='black')
ax1.hist(X[y == 0], bins=30, alpha=0.6, label='NG', color='red', edgecolor='black')
ax1.axvline(threshold_lr, color='blue', linestyle='--', linewidth=2, label=f'Logistic threshold: {threshold_lr:.3f}g')
ax1.axvline(threshold_lda, color='orange', linestyle='--', linewidth=2, label=f'LDA threshold: {threshold_lda:.3f}g')
ax1.set_xlabel('Weight (g)')
ax1.set_ylabel('Frequency')
ax1.set_title('Weight Distribution: OK vs NG')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Probability curves
ax2 = axes[1]
weight_range = np.linspace(X.min() - 0.2, X.max() + 0.2, 300).reshape(-1, 1)

proba_lr = lr.predict_proba(weight_range)[:, 1]
proba_lda = lda.predict_proba(weight_range)[:, 1]
proba_qda = qda.predict_proba(weight_range)[:, 1]

ax2.plot(weight_range, proba_lr, label='Logistic', linewidth=2)
ax2.plot(weight_range, proba_lda, label='LDA', linewidth=2)
ax2.plot(weight_range, proba_qda, label='QDA', linewidth=2)
ax2.axhline(0.5, color='black', linestyle=':', alpha=0.5)
ax2.axhline(0.95, color='green', linestyle=':', alpha=0.5, label='High Confidence (0.95)')
ax2.axhline(0.05, color='red', linestyle=':', alpha=0.5, label='Low Confidence (0.05)')
ax2.set_xlabel('Weight (g)')
ax2.set_ylabel('P(OK | weight)')
ax2.set_title('Classification Probability Curves')
ax2.set_ylim([-0.05, 1.05])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/claude/decision_boundaries.png', dpi=300, bbox_inches='tight')
print("✓ Visualization saved: /home/claude/decision_boundaries.png")
plt.show()

## 16. Final Recommendation Report

In [ ]:
print("\n" + "="*100)
print("FINAL RECOMMENDATIONS FOR SMART PACKAGING LINE")
print("="*100)

# Find best model(s)
best_model_idx = results['Accuracy'].idxmax()
best_model = results.loc[best_model_idx, 'Model']
best_accuracy = results.loc[best_model_idx, 'Accuracy']

print(f"\n✓ WINNER: {best_model}")
print(f"  Accuracy: {best_accuracy:.6f}")
print(f"  vs Baseline (Z-Score): +{(best_accuracy - zscore_acc)*100:.2f}%")

if best_accuracy >= SIX_SIGMA_TARGET:
    print(f"  Status: ✓ ACHIEVES SIX SIGMA")
else:
    gap = SIX_SIGMA_TARGET - best_accuracy
    print(f"  Status: ✗ Below Six Sigma by {gap*100:.4f}%")

# Top 3 recommendations
top_3 = results.dropna(subset=['Accuracy']).nlargest(3, 'Accuracy')

print(f"\n📊 TOP 3 MODELS:")
for rank, (idx, row) in enumerate(top_3.iterrows(), 1):
    print(f"\n  {rank}. {row['Model']}")
    print(f"     Accuracy: {row['Accuracy']:.6f}")
    print(f"     AUC-ROC: {row['AUC-ROC']:.6f}")
    print(f"     Six Sigma: {row['vs Six Sigma']}")

# Implementation roadmap
print(f"\n\n📋 IMPLEMENTATION ROADMAP (4 WEEKS):")
print(f"""
WEEK 1:
  ✓ Logistic Regression (DONE)
  ✓ LDA (DONE)
  ✓ QDA (DONE)
  → Decision: Use LDA if assumptions hold, else QDA

WEEK 2:
  □ Bayesian Logistic Regression (posterior predictive)
  □ Integrate top model into app.py
  □ Add P(OK|weight) to HMI
  □ Add AMBIGUOUS flag (0.4 < P(OK) < 0.6)

WEEK 3-4:
  □ Execute full 420-kit DOE validation
  □ Log all predictions from all models
  □ Post-analysis: compare actual vs predicted
  □ If drift detected: implement CUSUM monitoring
  □ Final documentation + presentation
""")

# Risk assessment
print(f"\n\n⚠️  RISK ASSESSMENT:")
if best_accuracy < SIX_SIGMA_TARGET:
    print(f"  • Current best model ({best_accuracy:.6f}) is below Six Sigma target")
    print(f"  • Likely causes: non-normal data, insufficient sample size, outliers")
    print(f"  • Mitigation: (1) Shapiro-Wilk test, (2) outlier analysis, (3) increase n to 500+")
else:
    print(f"  • ✓ All top models achieve Six Sigma")
    print(f"  • Risk: low. Proceed with implementation.")

print(f"\n" + "="*100)

## 17. Export Results to Database & Excel

In [ ]:
# Save comparison table
results_export = results[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'DPMO', 'vs Six Sigma']].copy()
results_export.to_csv('/home/claude/model_comparison_results.csv', index=False)
print("✓ Results exported: /home/claude/model_comparison_results.csv")

# Save to Excel with formatting
try:
    from openpyxl import Workbook
    from openpyxl.styles import PatternFill, Font, Alignment
    
    wb = Workbook()
    ws = wb.active
    ws.title = "Model Comparison"
    
    # Headers
    headers = results_export.columns.tolist()
    for col_num, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_num, value=header)
        cell.fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
        cell.font = Font(bold=True, color="FFFFFF")
        cell.alignment = Alignment(horizontal="center")
    
    # Data rows
    for row_num, row_data in enumerate(results_export.values, 2):
        for col_num, value in enumerate(row_data, 1):
            cell = ws.cell(row=row_num, column=col_num, value=value)
            
            # Color code Six Sigma column
            if col_num == len(headers) and value == '✓ PASS':
                cell.fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
                cell.font = Font(color="006100")
            elif col_num == len(headers) and value == '✗ FAIL':
                cell.fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
                cell.font = Font(color="9C0006")
            
            cell.alignment = Alignment(horizontal="center")
    
    # Adjust column widths
    ws.column_dimensions['A'].width = 25
    for col in ['B', 'C', 'D', 'E', 'F', 'G', 'H']:
        ws.column_dimensions[col].width = 15
    
    wb.save('/home/claude/model_comparison_results.xlsx')
    print("✓ Excel exported: /home/claude/model_comparison_results.xlsx")
except ImportError:
    print("(openpyxl not available for Excel export)")

## 18. Summary Statistics

In [ ]:
print("\n" + "="*100)
print("SUMMARY STATISTICS")
print("="*100)

print(f"\nData Summary:")
print(f"  Total kits: {len(y)}")
print(f"  OK kits: {np.sum(y)} ({100*np.sum(y)/len(y):.1f}%)")
print(f"  NG kits: {len(y) - np.sum(y)} ({100*(len(y)-np.sum(y))/len(y):.1f}%)")
print(f"  Recipes: R3, R4, R6")

print(f"\nModel Performance Summary:")
valid_results = results.dropna(subset=['Accuracy'])
print(f"  Mean accuracy: {valid_results['Accuracy'].mean():.6f}")
print(f"  Std deviation: {valid_results['Accuracy'].std():.6f}")
print(f"  Best accuracy: {valid_results['Accuracy'].max():.6f}")
print(f"  Worst accuracy: {valid_results['Accuracy'].min():.6f}")

print(f"\nSix Sigma Achievement:")
six_sigma_pass = (results['vs Six Sigma'] == '✓ PASS').sum()
six_sigma_total = len(results) - (results['vs Six Sigma'] == 'N/A').sum()
print(f"  Models achieving 99.9997%: {six_sigma_pass} / {six_sigma_total}")

print(f"\nTop Model Details:")
top_row = results.dropna(subset=['Accuracy']).iloc[0]
for col in ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']:
    val = top_row[col]
    if pd.isna(val):
        print(f"  {col}: N/A")
    else:
        print(f"  {col}: {val:.6f}")

print(f"\n" + "="*100)

In [ ]:
print("\n✅ ANALYSIS COMPLETE\n")
print("Files generated:")
print("  1. /home/claude/model_comparison_results.csv")
print("  2. /home/claude/model_comparison_results.xlsx")
print("  3. /home/claude/model_comparison.png")
print("  4. /home/claude/decision_boundaries.png")
print("\nNext steps:")
print("  • Review results with team")
print("  • Select best model(s) for production")
print("  • Integrate into app.py")
print("  • Execute 420-kit validation")